# 1. Machine Learning Objective

The previous analysis identified a positive relationship between electricity
demand and spot price, together with strong temporal patterns in both demand
and RRP.

This notebook applies machine-learning models to determine whether electricity
spot prices can be predicted using demand and time-related features.

The first task is formulated as a regression problem, where the target variable
is the continuous Regional Reference Price (RRP).

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [5]:
# Load the csv 
df = pd.read_csv(
    "../data/processed/clean_price_demand.csv", 
    parse_dates = ["SETTLEMENTDATE"]
)

df.head()

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE
0,NSW1,2026-08-01 00:05:00,8898.91,89.88,TRADE
1,NSW1,2026-08-01 00:10:00,9032.33,95.05,TRADE
2,NSW1,2026-08-01 00:15:00,8989.12,88.88,TRADE
3,NSW1,2026-08-01 00:20:00,9036.72,88.88,TRADE
4,NSW1,2026-08-01 00:25:00,8946.67,84.79,TRADE


In [7]:
# Create temporal features 
df["hour"] = df["SETTLEMENTDATE"].dt.hour
df["day_of_week"] = df["SETTLEMENTDATE"].dt.dayofweek
df["is_weekend"] =  (df["day_of_week"] >= 5).astype(int)

In [9]:
# Define features and targets
features = [
    "TOTALDEMAND",
    "hour",
    "day_of_week",
    "is_weekend"
]

X = df[features]
y = df["RRP"]

## 2. Chronological Train-Test Split

Because electricity-market observations are ordered through time, the dataset is
split chronologically rather than randomly.

The first 80% of observations are used for training, while the final 20% are
reserved for testing. This more closely reflects a real forecasting setting,
where historical observations are used to predict later market behaviour.

In [10]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

In [11]:
# Create simple benchmark
baseline_prediction = np.repeat(
    y_train.mean(),
    len(y_test)
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_prediction
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_prediction
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_prediction
)

print(f"Baseline MAE: {baseline_mae:.2f}")
print(f"Baseline RMSE: {baseline_rmse:.2f}")
print(f"Baseline R²: {baseline_r2:.3f}")

Baseline MAE: 36.15
Baseline RMSE: 45.86
Baseline R²: -0.070
